# Troubleshooting missing solutions

:::{autolink-concat}
:::

In [ ]:
from IPython.display import display

import qrules

When {func}`.generate_transitions` cannot find any transition, it raises a {class}`RuntimeError` that carries an {class}`.ExecutionInfo` object. Before that, it often emits a warning that some conservation rules were *not executed*. This page explains what that means, using the decay $J/\psi \to K^0 \Sigma^+ \bar{p}$ via a $\Delta(1910)^+$ resonance, with only strong interactions allowed.

In [ ]:
try:
    qrules.generate_transitions(
        initial_state="J/psi(1S)",
        final_state=["K0", "Sigma+", "p~"],
        allowed_intermediate_particles=["Delta(1910)+"],
        allowed_interaction_types="strong",
        formalism="canonical-helicity",
    )
except RuntimeError as error:
    execution_info = error.args[1]
    display(execution_info)

The {attr}`~.ExecutionInfo.not_executed_node_rules` and {attr}`~.ExecutionInfo.not_executed_edge_rules` list nearly every rule. This does **not** mean that these rules were skipped by mistake, nor that they are violated. The quantum numbers are found with a constraint solver that works by backtracking: as soon as one rule fails for every candidate value of a quantum number, the solver abandons that branch of the search, so rules that would have been checked later are never called. A rule can also remain unexecuted if one of the quantum numbers it requires has no domain in the {class}`.EdgeSettings` or {class}`.NodeSettings`.

The {attr}`~.ExecutionInfo.violated_node_rules` are of more help, but they are intersected over all problem sets. The reaction is split into many problem sets, one for each topology and each combination of interaction settings such as $L$ and $S$, and a rule is only reported if it is violated in all of them. Here, the problem sets fail on different rules, so no single rule is reported.

A more practical way to find the cause is to relax the problem. The $J/\psi$ has isospin 0, while $\Delta^+\bar{p}$ can only couple to isospin 1 or 2, so isospin conservation forbids the first decay in the strong interaction. Allowing electromagnetic interactions, which do not conserve isospin, results in solutions:

In [ ]:
reaction = qrules.generate_transitions(
    initial_state="J/psi(1S)",
    final_state=["K0", "Sigma+", "p~"],
    allowed_intermediate_particles=["Delta(1910)+"],
    allowed_interaction_types=["strong", "EM"],
    formalism="canonical-helicity",
)
len(reaction.transitions)

Use {meth}`.StateTransitionManager.set_allowed_interaction_types` or remove rules from the {class}`.NodeSettings` to narrow down which rule excludes a decay. See {doc}`/usage/conservation` for how the individual rules can be checked by hand.